# HW3 airline-tweet sentiment classification

This notebook runs `main.py` for the full experiment. It uses the second grouped outer fold as the local holdout, and compares the plain three-BERT ensemble, corrected heads, tweet-format adjustment, and second reading on those same rows. The model fits and calibrates on labeled course data outside the holdout. The benchmark is used for inference.

In Colab, select a GPU runtime. Start from the updated project folder or upload the files requested by the setup cell. Set `RUNS_DIR` to a mounted Google Drive path to keep checkpoints after a Colab runtime ends. Run the notebook from the project or its parent directory when using a local Jupyter session.

In [ ]:
from pathlib import Path

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

PROJECT_DIR = None  # Path to IAI_HW3. None checks common locations.
RUNS_DIR = None  # Example: "/content/drive/MyDrive/HW3_runs"
STUDENT_ID = "113511195"  # From the earlier course package filenames.
OUTER_FOLD = 2  # One-based.
RUN_OOF = False  # Fits three models in each outer fold.
DOWNLOAD_IN_COLAB = False  # Set True to download the final ZIP and CSV.
ON_COLAB = colab_files is not None
INSTALL_DEPS = ON_COLAB
DEVICE = "cuda" if ON_COLAB else "auto"

In [ ]:
from contextlib import chdir
import shlex
import subprocess
import sys

def is_project(path):
    return all(
        (path / relative).is_file()
        for relative in ("main.py", "dataset/dataset.csv", "dataset/benchmark.csv")
    )

if ON_COLAB and any(
    value is not None and Path(value).is_relative_to("/content/drive")
    for value in (PROJECT_DIR, RUNS_DIR)
):
    from google.colab import drive
    drive.mount("/content/drive")

if PROJECT_DIR is None:
    candidates = (
        Path.cwd(),
        Path.cwd() / "IAI_HW3",
        Path.cwd() / "Intro_to_AI_HW" / "IAI_HW3",
    )
    project = next((path for path in candidates if is_project(path)), None)
else:
    project = Path(PROJECT_DIR).expanduser().resolve()

if project is None and ON_COLAB:
    print("Upload the updated main.py, IAI_HW3.ipynb, requirements.txt, dataset.csv and benchmark.csv.")
    project = Path("/content/IAI_HW3")
    (project / "dataset").mkdir(parents=True, exist_ok=True)
    with chdir(project):
        uploaded = colab_files.upload()
    for name, payload in uploaded.items():
        filename = Path(name).name
        if filename in {"dataset.csv", "benchmark.csv"}:
            (project / "dataset" / filename).write_bytes(payload)
    del uploaded

if project is None or not is_project(project):
    raise FileNotFoundError("Set PROJECT_DIR to the folder with main.py and both course CSV files.")
project = project.resolve()
data_dir = project / "dataset"
runs = (Path(RUNS_DIR).expanduser() if RUNS_DIR is not None else project / "runs").resolve()
runs.mkdir(parents=True, exist_ok=True)
holdout_dir = runs / f"holdout_fold_{OUTER_FOLD:02d}"
oof_dir = runs / "oof"
delivery_dir = runs / "delivery"

if INSTALL_DEPS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")],
        cwd=project,
        check=True,
    )

def command(action, *options):
    return [
        sys.executable, str(project / "main.py"), action, "--data-dir", str(data_dir),
        *(str(option) for option in options),
    ]

def show_command(args):
    print(subprocess.list2cmdline(args) if sys.platform == "win32" else shlex.join(args), flush=True)

def run(action, *options):
    args = command(action, *options)
    show_command(args)
    subprocess.run(args, cwd=project, check=True)

print(f"Project {project}")
print(f"Runs {runs}")

## Prepare the grouped splits

The outer fold isolates the local evaluation rows. Each member uses its own stopping set inside the remaining labeled rows. Identical normalized tweets stay together wherever a split is made.

In [ ]:
run("prepare", "--run-dir", holdout_dir, "--outer-fold", OUTER_FOLD)

## Train the three BERT members

Each member runs four masked-language-model adaptation epochs on its fitting tweets, then fine-tunes BERT for sentiment with exponential moving average checkpoints. The stopping-set logits and features are used to fit the corrected heads and tweet-format adjustment.

In [ ]:
import torch

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running the training cell.")
run("train", "--run-dir", holdout_dir, "--outer-fold", OUTER_FOLD, "--device", DEVICE)

## Compare all four stages on the holdout

The report includes macro-F1, neutral F1, confusion matrices, and mistakes fixed or introduced as each stage is added. Read the labeled changed-prediction example alongside the aggregate metrics. The reference local holdout macro-F1 was 0.91026. The reference scores on Kaggle were 0.9190 after calibration and 0.9218 after the second reading. The benchmark score for this run will come from Kaggle.

In [ ]:
import json
from IPython.display import Image, JSON, display

run("evaluate", "--run-dir", holdout_dir, "--device", DEVICE)
display(JSON(data=json.loads((holdout_dir / "report.json").read_text(encoding="utf-8")), expanded=True))
for plot in sorted(holdout_dir.rglob("*.png")):
    print(plot.relative_to(holdout_dir))
    display(Image(filename=str(plot)))

## Optional 10-fold out-of-fold evaluation

Set `RUN_OOF = True` to evaluate all labeled rows. Each outer fold fits its own learned components with only that fold's outer training rows. Fold 2 can reuse the matching holdout run. The reference second reading changed pooled macro-F1 from 0.89931 to 0.90045 and neutral F1 from 0.82380 to 0.82491.

In [ ]:
oof_options = (
    "--run-dir", oof_dir, "--reuse-holdout", holdout_dir, "--device", DEVICE,
)
if RUN_OOF:
    run("oof", *oof_options)
    display(JSON(data=json.loads((oof_dir / "report.json").read_text(encoding="utf-8")), expanded=True))
else:
    show_command(command("oof", *oof_options))

## Generate and verify the submission files

Predict the benchmark in its original order. The packaging command reloads the packaged checkpoints and reproduces the submission. The notebook also compares the packaged CSV with the direct prediction, checks the header and ID sequence, and verifies the allowed labels. Submit the ZIP to E3 and upload the CSV to Kaggle yourself.

In [ ]:
import csv

direct_predictions = holdout_dir / "benchmark_predictions.csv"
run("predict", "--run-dir", holdout_dir, "--output", direct_predictions, "--device", DEVICE)
run(
    "package", "--run-dir", holdout_dir, "--student-id", STUDENT_ID,
    "--output", delivery_dir, "--device", DEVICE,
)

submission = delivery_dir / "predictions.csv"
archive = delivery_dir / f"HW3_{STUDENT_ID}.zip"
if direct_predictions.read_bytes() != submission.read_bytes():
    raise ValueError("Direct and packaged predictions differ.")
with (data_dir / "benchmark.csv").open(encoding="utf-8-sig", newline="") as file:
    benchmark_ids = [row["id"] for row in csv.DictReader(file)]
with submission.open(encoding="utf-8", newline="") as file:
    predictions = csv.DictReader(file)
    if predictions.fieldnames != ["id", "label"]:
        raise ValueError(f"Unexpected CSV header {predictions.fieldnames}")
    rows = list(predictions)
submission_ids = [row["id"] for row in rows]
if submission_ids != benchmark_ids or len(set(submission_ids)) != len(submission_ids):
    raise ValueError("Submission IDs differ from the unique benchmark IDs in CSV order.")
if any(row["label"] not in {"negative", "neutral", "positive"} for row in rows):
    raise ValueError("The submission contains a label outside the three allowed sentiments.")
if not archive.is_file():
    raise FileNotFoundError(archive)
print(f"Validated {len(rows):,} predictions.")
print(f"E3 package {archive}")
print(f"Kaggle submission {submission}")
if ON_COLAB and DOWNLOAD_IN_COLAB:
    colab_files.download(str(submission))
    colab_files.download(str(archive))